# Lesson 5 — query & join · ค้นและเชื่อมตาราง

`where` ของ LanceDB รับ SQL แค่ส่วน filter ไม่มี JOIN ไม่มี GROUP BY
บทนี้ทำ query ที่มีให้ก่อน แล้วค่อยดูว่าพอต้อง join จะทำยังไง
คำตอบสั้น ๆ คือ ดึงเป็น Arrow แล้วให้ pandas หรือ DuckDB ทำต่อ

In [1]:
%pip install -q lancedb pandas duckdb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import lancedb
import pandas as pd
from IPython.display import display

db = lancedb.connect("./data")

users = db.create_table("users", data=[
    {"id": 1, "name": "nat",  "plan": "team"},
    {"id": 2, "name": "beta", "plan": "pro"},
    {"id": 4, "name": "odin", "plan": "free"},
], mode="overwrite")

orders = db.create_table("orders", data=[
    {"order_id": 10, "user_id": 1, "amount": 300},
    {"order_id": 11, "user_id": 1, "amount": 120},
    {"order_id": 12, "user_id": 2, "amount": 80},
    {"order_id": 13, "user_id": 9, "amount": 50},  # user 9 does not exist
], mode="overwrite")

print("users"); display(users.to_pandas())
print("orders"); display(orders.to_pandas())

users


[2026-09-10T11:47:08Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/05-query-join/data/users.lance, it will be created
[2026-09-10T11:47:08Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/05-query-join/data/orders.lance, it will be created


,id,name,plan
0,1,nat,team
1,2,beta,pro
2,4,odin,free


orders


,order_id,user_id,amount
0,10,1,300
1,11,1,120
2,12,2,80
3,13,9,50


**Filter** `where` รับ `=` `>` `AND` `OR` `IN` `LIKE` `IS NULL`
`select` เลือกเฉพาะ column ที่ต้องการ อ่านน้อยลง เร็วขึ้น
`limit` จำเป็นเมื่อไม่มี vector search ไม่ใส่จะได้ค่า default 10

เงื่อนไข: plan เป็น pro หรือ team **และ** ชื่อขึ้นต้น n → เหลือ nat คนเดียว (beta ตก LIKE, odin ตก IN)

In [3]:
users.search().where("plan IN ('pro', 'team') AND name LIKE 'n%'").select(["id", "name", "plan"]).limit(10).to_pandas()

,id,name,plan
0,1,nat,team


`amount > 100` → order 10 (300) กับ 11 (120) · order 12 (80) และ 13 (50) ตก

In [4]:
orders.search().where("amount > 100").limit(10).to_pandas()

,order_id,user_id,amount
0,10,1,300
1,11,1,120


**Join แบบที่ 1 — pandas**
ดึงสองตารางออกมาเป็น DataFrame แล้ว `merge` ด้วย `user_id = id`
`how="left"` เก็บทุก order แม้ไม่เจอ user → order 13 ได้ name เป็น NaN
เหมาะกับตารางเล็ก ข้อมูลทั้งหมดขึ้น memory

In [5]:
u = users.to_pandas()
o = orders.to_pandas()
o.merge(u, left_on="user_id", right_on="id", how="left")[["order_id", "user_id", "name", "plan", "amount"]]

,order_id,user_id,name,plan,amount
0,10,1,nat,team,300
1,11,1,nat,team,120
2,12,2,beta,pro,80
3,13,9,NaN,NaN,50


**Join แบบที่ 2 — DuckDB**
DuckDB อ่าน Arrow table ได้ตรง ๆ เขียน SQL เต็มรูปแบบได้เลย
JOIN · GROUP BY · window function ครบ ไม่ต้องแปลงอะไร
ตัวแปร Python ที่เป็น Arrow table ใช้ชื่อใน SQL ได้ทันที

ผลที่ควรได้: nat 2 orders รวม 420 · beta 1 order 80 · odin 0 order total NULL

In [6]:
import duckdb

users_arrow = users.to_arrow()
orders_arrow = orders.to_arrow()

duckdb.sql("""
    SELECT u.name, u.plan, COUNT(o.order_id) AS orders, SUM(o.amount) AS total
    FROM users_arrow u
    LEFT JOIN orders_arrow o ON o.user_id = u.id
    GROUP BY u.name, u.plan
    ORDER BY total DESC NULLS LAST
""").df()

,name,plan,orders,total
0,nat,team,2,420.0
1,beta,pro,1,80.0
2,odin,free,0,NaN


**เช็คด้วยมือ** — ผลรวมของ nat มาจากไหน
กรอง orders ที่ user_id = 1 แล้วบวก: 300 + 120 = 420 ตรงกับ `total` ข้างบน

In [7]:
nat_orders = o[o.user_id == 1][["order_id", "amount"]].copy()
nat_orders.loc["sum"] = ["", nat_orders.amount.sum()]
nat_orders

,order_id,amount
0,10,300
1,11,120
sum,,420


**Orphan** order 13 ชี้ไป user 9 ที่ไม่มีอยู่
LanceDB ไม่มี foreign key ไม่มีใครห้ามตอน insert
หาเจอทีหลังด้วย LEFT JOIN แล้วดูว่าฝั่ง user เป็น NULL
ความสัมพันธ์ระหว่างตาราง เป็นหน้าที่ของโค้ดฝั่งเรา ไม่ใช่ของ DB

In [8]:
duckdb.sql("""
    SELECT o.order_id, o.user_id, o.amount, u.id AS matched_user
    FROM orders_arrow o
    LEFT JOIN users_arrow u ON o.user_id = u.id
    WHERE u.id IS NULL
""").df()

,order_id,user_id,amount,matched_user
0,13,9,50,<NA>
